# Marketplace Buyer-Seller Analysis

## Project Overview
Analyze marketplace activity data to compare buyer behavior, seller performance, liquidity patterns, and transaction balance.

## Learning Objectives
- Analyze two-sided marketplace dynamics (buyer and seller perspectives)
- Calculate marketplace liquidity and supply-demand balance metrics
- Compare seller performance tiers and revenue concentration
- Analyze buyer purchase frequency and lifetime value distribution
- Identify marketplace health indicators and risk signals

## Problem Statement
A two-sided marketplace needs to understand the balance between buyers and sellers. Too few sellers means poor selection; too few buyers means seller churn. We analyze transaction data to assess marketplace health and identify growth levers.

## Why This Project Matters
Two-sided marketplaces (Etsy, Airbnb, Uber) live or die by network effects. Understanding both sides — and the balance between them — is critical for growth strategy, pricing, and operational decisions.

## Dataset Overview
Synthetic marketplace data: ~12,000 transactions over 12 months with 2,000 buyers and 500 sellers across 5 categories.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_txn = 12000
n_buyers = 2000
n_sellers = 500

dates = pd.date_range('2023-01-01', periods=365, freq='D')
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Toys', 'Sports']
cat_weights = np.array([0.30, 0.25, 0.20, 0.15, 0.10])

# Seller quality tiers
seller_ids = np.arange(n_sellers)
seller_tier = np.random.choice(['top', 'mid', 'low'], size=n_sellers, p=[0.10, 0.40, 0.50])
seller_cat = np.random.choice(categories, size=n_sellers, p=cat_weights)

# Buyer purchase frequency follows power law
buyer_ids = np.arange(n_buyers)
buyer_freq = np.random.pareto(1.5, n_buyers) + 1  # heavy tail
buyer_freq = np.clip(buyer_freq, 1, 50).astype(int)

rows = []
for _ in range(n_txn):
    # Pick buyer weighted by frequency
    b_weights = buyer_freq / buyer_freq.sum()
    buyer = np.random.choice(buyer_ids, p=b_weights)

    # Pick seller weighted by tier
    tier_mult = {'top': 5.0, 'mid': 2.0, 'low': 1.0}
    s_weights = np.array([tier_mult[seller_tier[s]] for s in seller_ids])
    s_weights /= s_weights.sum()
    seller = np.random.choice(seller_ids, p=s_weights)

    category = seller_cat[seller]
    date = np.random.choice(dates)

    # Price by category and seller tier
    base_price = {'Electronics': 120, 'Clothing': 45, 'Home & Garden': 65,
                  'Toys': 30, 'Sports': 55}[category]
    tier_mod = {'top': 1.2, 'mid': 1.0, 'low': 0.8}[seller_tier[seller]]
    price = max(5, round(np.random.lognormal(np.log(base_price * tier_mod), 0.5), 2))

    # Rating
    tier_rating = {'top': 4.5, 'mid': 3.8, 'low': 3.2}[seller_tier[seller]]
    rating = np.clip(round(np.random.normal(tier_rating, 0.5), 1), 1.0, 5.0)

    rows.append({
        'txn_id': len(rows), 'date': date, 'buyer_id': buyer, 'seller_id': seller,
        'category': category, 'price': price, 'rating': rating,
        'seller_tier': seller_tier[seller]
    })

df = pd.DataFrame(rows)
df['month'] = df['date'].dt.to_period('M')
df['marketplace_fee'] = (df['price'] * 0.12).round(2)  # 12% take rate

print(f'Dataset: {df.shape}')
print(f'Unique buyers: {df["buyer_id"].nunique()}')
print(f'Unique sellers: {df["seller_id"].nunique()}')
print(f'Total GMV: ${df["price"].sum():,.0f}')
print(f'Total marketplace revenue: ${df["marketplace_fee"].sum():,.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nCategory distribution:\n{df["category"].value_counts()}')
print(f'\nSeller tier distribution:\n{df["seller_tier"].value_counts()}')
print(f'\nPrice stats:')
print(df['price'].describe().round(2))

## Marketplace GMV Trend

In [ ]:
monthly_gmv = df.groupby('month').agg(
    gmv=('price', 'sum'),
    txns=('txn_id', 'count'),
    active_buyers=('buyer_id', 'nunique'),
    active_sellers=('seller_id', 'nunique')
).reset_index()
monthly_gmv['month_str'] = monthly_gmv['month'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].bar(monthly_gmv['month_str'], monthly_gmv['gmv'], edgecolor='black', color='steelblue')
axes[0].set_title('Monthly GMV')
axes[0].set_ylabel('GMV ($)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly_gmv['month_str'], monthly_gmv['active_buyers'], marker='o', label='Active Buyers')
axes[1].plot(monthly_gmv['month_str'], monthly_gmv['active_sellers'], marker='s', label='Active Sellers')
axes[1].set_title('Active Users by Month')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gmv_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seller Performance Tiers

In [ ]:
seller_stats = df.groupby('seller_id').agg(
    txns=('txn_id', 'count'),
    revenue=('price', 'sum'),
    avg_rating=('rating', 'mean'),
    categories=('category', 'nunique'),
    tier=('seller_tier', 'first')
).reset_index()

tier_summary = seller_stats.groupby('tier').agg(
    sellers=('seller_id', 'count'),
    avg_txns=('txns', 'mean'),
    avg_revenue=('revenue', 'mean'),
    avg_rating=('avg_rating', 'mean')
).round(1)
print('Seller tier summary:')
print(tier_summary)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in zip(axes, ['avg_txns', 'avg_revenue', 'avg_rating'],
                           ['Avg Transactions', 'Avg Revenue ($)', 'Avg Rating']):
    tier_summary[col].plot.bar(ax=ax, edgecolor='black', color=['#F44336', '#FF9800', '#4CAF50'])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seller_tiers.png'), dpi=100, bbox_inches='tight')
plt.show()

## Revenue Concentration (Pareto)

In [ ]:
seller_rev = seller_stats.sort_values('revenue', ascending=False)
seller_rev['cum_pct'] = seller_rev['revenue'].cumsum() / seller_rev['revenue'].sum() * 100
seller_rev['seller_pct'] = np.arange(1, len(seller_rev) + 1) / len(seller_rev) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(seller_rev['seller_pct'], seller_rev['cum_pct'], linewidth=2, color='steelblue')
ax.axhline(80, color='red', linestyle='--', alpha=0.7, label='80% of revenue')
ax.axvline(20, color='green', linestyle='--', alpha=0.7, label='20% of sellers')
ax.set_title('Seller Revenue Concentration (Pareto Curve)')
ax.set_xlabel('% of Sellers (ranked by revenue)')
ax.set_ylabel('Cumulative % of Revenue')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pareto_curve.png'), dpi=100, bbox_inches='tight')
plt.show()

top20_rev = seller_rev.head(int(len(seller_rev) * 0.2))['revenue'].sum()
total_rev = seller_rev['revenue'].sum()
print(f'Top 20% of sellers generate {top20_rev/total_rev*100:.1f}% of revenue')

## Buyer Behavior Analysis

In [ ]:
buyer_stats = df.groupby('buyer_id').agg(
    purchases=('txn_id', 'count'),
    total_spend=('price', 'sum'),
    avg_order=('price', 'mean'),
    categories=('category', 'nunique'),
    unique_sellers=('seller_id', 'nunique')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(buyer_stats['purchases'], bins=range(1, 30), edgecolor='black', color='steelblue')
axes[0].set_title('Purchase Frequency Distribution')
axes[0].set_xlabel('Purchases')
axes[0].set_ylabel('Buyers')

axes[1].hist(buyer_stats['total_spend'], bins=50, edgecolor='black', color='#FF9800')
axes[1].set_title('Total Spend Distribution')
axes[1].set_xlabel('Total Spend ($)')
axes[1].set_ylabel('Buyers')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'buyer_behavior.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Median purchases per buyer: {buyer_stats["purchases"].median():.0f}')
print(f'Median total spend: ${buyer_stats["total_spend"].median():.0f}')
print(f'Avg categories shopped: {buyer_stats["categories"].mean():.1f}')
# Repeat buyer rate
repeat = (buyer_stats['purchases'] > 1).mean() * 100
print(f'Repeat buyer rate: {repeat:.1f}%')

## Category Analysis

In [ ]:
cat_stats = df.groupby('category').agg(
    txns=('txn_id', 'count'),
    gmv=('price', 'sum'),
    avg_price=('price', 'mean'),
    avg_rating=('rating', 'mean'),
    unique_sellers=('seller_id', 'nunique'),
    unique_buyers=('buyer_id', 'nunique')
).round(1)
cat_stats['buyer_seller_ratio'] = (cat_stats['unique_buyers'] / cat_stats['unique_sellers']).round(1)
print('Category analysis:')
print(cat_stats)

fig, ax = plt.subplots(figsize=(10, 5))
cat_stats[['gmv']].plot.bar(ax=ax, edgecolor='black', color='teal', legend=False)
ax.set_title('GMV by Category')
ax.set_ylabel('GMV ($)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'category_gmv.png'), dpi=100, bbox_inches='tight')
plt.show()

## Marketplace Liquidity

In [ ]:
# Buyer-to-seller ratio over time
liquidity = monthly_gmv.copy()
liquidity['bs_ratio'] = (liquidity['active_buyers'] / liquidity['active_sellers']).round(1)
liquidity['avg_txn_value'] = (liquidity['gmv'] / liquidity['txns']).round(0)

print('Monthly liquidity metrics:')
print(liquidity[['month_str', 'active_buyers', 'active_sellers', 'bs_ratio', 'avg_txn_value']])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(liquidity['month_str'], liquidity['bs_ratio'], edgecolor='black', color='#9C27B0')
ax.set_title('Buyer-to-Seller Ratio Over Time')
ax.set_ylabel('Buyers per Seller')
ax.tick_params(axis='x', rotation=45)
ax.axhline(liquidity['bs_ratio'].mean(), color='red', linestyle='--', label=f'Avg: {liquidity["bs_ratio"].mean():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'liquidity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Revenue is highly concentrated** — top 20% of sellers generate ~70-80% of GMV (power law dynamics)
- **Top-tier sellers** outperform mid-tier by 3-5× in transactions and revenue — invest in seller success programs
- **Repeat buyer rate** is a key health metric — higher repeat rate indicates marketplace stickiness
- **Electronics** dominates GMV but has fewer sellers per buyer — supply expansion opportunity
- **Buyer-to-seller ratio** is stable — marketplace is in balance, neither side is starved
- **Low-tier sellers** have poor ratings and low transaction volume — consider quality standards or seller education
- **Category diversification** among buyers is moderate — cross-category recommendation could increase wallet share

## Limitations
- Synthetic data with simplified transaction model
- No return/refund data
- No seller acquisition cost or buyer CAC data
- No seasonal promotion effects
- No geographic dimension
- No search/discovery funnel data

## How to Improve This Project
- Add return/refund analysis for net revenue calculation
- Include seller and buyer acquisition costs for unit economics
- Add search-to-purchase funnel analysis
- Build seller churn prediction model
- Include geographic supply-demand mapping

## Production Considerations
- Real-time marketplace health dashboard (GMV, liquidity, take rate)
- Automated seller quality monitoring and interventions
- Dynamic take rate optimization by category
- Buyer-seller matching and recommendation engine
- Fraud detection on both sides of the marketplace

## Common Mistakes
- Optimizing for one side of the marketplace at the expense of the other
- Measuring GMV growth without tracking unit economics
- Ignoring revenue concentration risk (top seller dependency)
- Not tracking buyer-to-seller ratio as a health metric
- Treating all categories equally when they have very different dynamics

## Mini Challenge / Exercises
1. Calculate the Gini coefficient for seller revenue — how unequal is the distribution?
2. What is the marketplace take rate, and how does it vary by category?
3. If the top 5 sellers left, what percentage of GMV would be lost?
4. Build a simple cohort analysis of buyer retention (% of month-1 buyers still active in month 6).

## Final Summary / Key Takeaways
- Two-sided marketplaces require balanced analysis of both buyers and sellers
- Revenue concentration is a feature and a risk — top sellers are valuable but create dependency
- Liquidity (buyer-to-seller ratio) is the fundamental marketplace health metric
- Category-level analysis reveals different dynamics that need different strategies
- Seller quality directly drives marketplace value — invest in seller success
- This is the 100th and final data analysis project — congratulations on completing the full series!